In [2]:
%load_ext cuml.accel
%run /workspace/alvin/SAR_ML/notebooks/SSR/SSRtransforms_preloaded.py
# %run /mnt/d/Users/Admin/Projects/dso/SAR_ML/notebooks/SSR/SSRtransforms_preloaded.py
import os
import random
from collections import defaultdict
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms, models
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, ConcatDataset, Subset, Dataset
import re
import copy
import torchvision
import numpy as np
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from tqdm import tqdm

/opt/py_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def set_seed(seed=42):
    """Set all random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Uncomment only if you need 100% determinism and can handle errors
    # torch.use_deterministic_algorithms(True, warn_only=True)
    # os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    
    os.environ["PYTHONHASHSEED"] = str(seed)

def worker_init_fn(worker_id):
    """DataLoader worker init for reproducibility"""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

In [4]:
workspace = "/workspace/alvin/SAR_ML"
# workspace = "/mnt/d/Users/Admin/Projects/dso/SAR_ML"
data_workspace = os.path.join(workspace, "data/SAMPLE")

# SSR w noise

In [4]:
gmm_cache = build_gmm_cache(input_dir = os.path.join(data_workspace, "mat_files/synth"), processing_func = LogMapping(c = 1000.0))

Found 1345 .mat files


Fitting GMMs: 100%|███████████████████████████████████████████████████████████████████| 1345/1345 [01:01<00:00, 21.95it/s]


In [5]:
synth_ds = DatasetFolderWithPath(
    os.path.join(data_workspace, "mat_files/synth"),
    extensions = (".mat"),
    transform = transforms.Compose([Magnitude(), LogMapping(c = 1000.0), SSRAugmentation(gmm_cache, alpha=0.6, beta=0.4, apply_prob=0.5, gaussian_noise = True, mu_s = 0.0, sigma_s = 0.3), NumpyToTensor3Channel()]),
    loader = mat_file_loader
)

meas_ds = DatasetFolderWithPath(
    os.path.join(data_workspace, "mat_files/real"),
    extensions = (".mat"),
    transform = transforms.Compose([Magnitude(), LogMapping(c = 1000.0), NumpyToTensor3Channel()]),
    loader = mat_file_loader
)

In [6]:
synth_ds.classes

['2s1', 'bmp2', 'btr70', 'm1', 'm2', 'm35', 'm548', 'm60', 't72', 'zsu23']

In [17]:
excluded_label = 9
excluded_label_name = synth_ds.classes[excluded_label]
print(f"Excluding label {excluded_label} ({excluded_label_name}) from synthetic dataset")
new_synth_ds = RemappedSubset(synth_ds, exclude_label=excluded_label)

Excluding label 9 (zsu23) from synthetic dataset


In [18]:
train_ds = new_synth_ds
test_ds = meas_ds

ds_dict = {"train" : train_ds, "test": test_ds}
dataset_sizes = {"train" : len(train_ds), "test": len(test_ds)}

In [19]:
seed_lst = [42]

train_loss = []
# val_loss = []
train_acc =[]
# val_acc = []

for i, seed in enumerate(seed_lst):
    print(f"Training Run {i}: seed {seed}")

    set_seed(seed)

    dataloaders = {
        "train": DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=12, pin_memory = True, persistent_workers=True, worker_init_fn=worker_init_fn, generator=torch.Generator().manual_seed(seed)),
        # "val": DataLoader(valid_ds, batch_size=16, shuffle=False, num_workers=8, pin_memory = True, persistent_workers=True, worker_init_fn=worker_init_fn, generator=torch.Generator().manual_seed(seed)),
        "test": DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=12, pin_memory = True, persistent_workers=True, worker_init_fn=worker_init_fn, generator=torch.Generator().manual_seed(seed))
    }
    # load pre-trained model
    model = models.resnet18(weights = None)

    # Replace final layer for the number of classes
    model.fc = nn.Sequential(
        nn.Dropout(p = 0.4),
        nn.Linear(model.fc.in_features, len(synth_ds.class_to_idx) - 1)
    )
    
    # Define the loss function and optimizer
    criterion = nn.CrossEntropyLoss() # most common used nn for classification problems
    
    optimizer = optim.AdamW(model.parameters(), lr = 3e-4, weight_decay = 2e-4)
    
    scheduler = CosineAnnealingLR(optimizer, T_max = 200, eta_min = 3e-7)
        
    # move model to GPU
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    
    history = {
        "train_loss": [],
        # "val_loss" : [],
        "train_acc": [],
        # "val_acc" : []
    }
    
    # Training loops
    num_epochs = 200
    for epoch in range(num_epochs):
        print(f"Epoch {epoch}")
        if epoch == 0:
            print(f"First layer mean: {model.conv1.weight.data.mean():.6f}")
        for phase in ["train"]:
            if phase == "train":
                model.train()
            else:
                model.eval()
    
            running_loss = 0.0
            running_corrects = 0 # correct predictions
    
            for inputs, labels in tqdm(dataloaders[phase], leave = False):
                inputs = inputs.to(device)
                labels = labels.to(device)
    
                optimizer.zero_grad() # clear the gradient from previous iteration
    
                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels) # check if output and labels match
    
                    if phase == "train":
                        loss.backward()
                        optimizer.step()
                        # scheduler.step() # scheduler here if OneCycleLR
    
                running_loss += loss.item() * inputs.size(0)
                running_corrects += (preds == labels).sum().item()
    
            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects / dataset_sizes[phase]
            
            history[f"{phase}_loss"].append(epoch_loss)
            history[f"{phase}_acc"].append(epoch_acc)
    
            print(f"{phase} Loss: {epoch_loss:.7f} Acc: {epoch_acc:.7f}")
    
        scheduler.step()
        print(f"Epoch {epoch} LR: {scheduler.get_last_lr()[0]:.10f}")

        # if epoch % 15 == 0:
        #     torch.save({
        #         "epoch": epoch,
        #         "model_state_dict": model.state_dict(),
        #         "optimizer_state_dict": optimizer.state_dict(),
        #         "scheduler_state_dict": scheduler.state_dict(),
        #         "loss": epoch_loss,
        #         "history": history,
        #     }, os.path.join(workspace, f"weights/SSR/Experiment_2/seed{seed}_epoch{epoch}.pth"))
        
    print("Training complete!")
    
    train_loss.append(history["train_loss"])
    # val_loss.append(history["val_loss"])
    train_acc.append(history["train_acc"])
    # val_acc.append(history["val_acc"])
    
    torch.save(model.state_dict(), os.path.join(workspace, f"weights/SSR/OOD/rn18_seed{seed}_b16_rm_{excluded_label_name}.pth"))

train_loss = np.array(train_loss)
# val_loss = np.array(val_loss)
train_acc = np.array(train_acc)
# val_acc = np.array(val_acc)

Training Run 0: seed 42
Epoch 0
First layer mean: -0.000035


train Loss: 1.6634328 Acc: 0.3706234
Epoch 0 LR: 0.0002999815
Epoch 1


train Loss: 0.6085531 Acc: 0.7993168
Epoch 1 LR: 0.0002999261
Epoch 2


train Loss: 0.3376454 Acc: 0.8872758
Epoch 2 LR: 0.0002998336
Epoch 3


train Loss: 0.1493407 Acc: 0.9573015
Epoch 3 LR: 0.0002997043
Epoch 4


...

train Loss: 0.0003994 Acc: 1.0000000
Epoch 195 LR: 0.0000005957
Epoch 196


train Loss: 0.0001594 Acc: 1.0000000
Epoch 196 LR: 0.0000004664
Epoch 197


train Loss: 0.0001115 Acc: 1.0000000
Epoch 197 LR: 0.0000003739
Epoch 198


train Loss: 0.0002930 Acc: 1.0000000
Epoch 198 LR: 0.0000003185
Epoch 199


train Loss: 0.0032953 Acc: 0.9991460
Epoch 199 LR: 0.0000003000
Training complete!


# wo aug

In [5]:
synth_ds = DatasetFolderWithPath(
    os.path.join(data_workspace, "mat_files/synth"),
    extensions = (".mat"),
    transform = transforms.Compose([Magnitude(), LogMapping(c = 1000.0), NumpyToTensor3Channel()]),
    loader = mat_file_loader
)

meas_ds = DatasetFolderWithPath(
    os.path.join(data_workspace, "mat_files/real"),
    extensions = (".mat"),
    transform = transforms.Compose([Magnitude(), LogMapping(c = 1000.0), NumpyToTensor3Channel()]),
    loader = mat_file_loader
)

In [6]:
for j in range(10):

    print(len(synth_ds))
    
    excluded_label = j
    excluded_label_name = synth_ds.classes[excluded_label]
    print(f"Excluding label {excluded_label} ({excluded_label_name}) from synthetic dataset")
    new_synth_ds = RemappedSubset(synth_ds, exclude_label=excluded_label)

    train_ds = new_synth_ds
    test_ds = meas_ds

    ds_dict = {"train" : train_ds, "test": test_ds}
    dataset_sizes = {"train" : len(train_ds), "test": len(test_ds)}

    seed_lst = [42]

    train_loss = []
    # val_loss = []
    train_acc =[]
    # val_acc = []

    for i, seed in enumerate(seed_lst):
        print(f"Training Run {i}: seed {seed}")

        set_seed(seed)

        dataloaders = {
            "train": DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=12, pin_memory = True, persistent_workers=True, worker_init_fn=worker_init_fn, generator=torch.Generator().manual_seed(seed)),
            # "val": DataLoader(valid_ds, batch_size=16, shuffle=False, num_workers=8, pin_memory = True, persistent_workers=True, worker_init_fn=worker_init_fn, generator=torch.Generator().manual_seed(seed)),
            "test": DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=12, pin_memory = True, persistent_workers=True, worker_init_fn=worker_init_fn, generator=torch.Generator().manual_seed(seed))
        }
        # load pre-trained model
        model = models.resnet18(weights = None)

        # Replace final layer for the number of classes
        model.fc = nn.Sequential(
            nn.Dropout(p = 0.4),
            nn.Linear(model.fc.in_features, len(synth_ds.class_to_idx) - 1)
        )
        
        # Define the loss function and optimizer
        criterion = nn.CrossEntropyLoss() # most common used nn for classification problems
        
        optimizer = optim.AdamW(model.parameters(), lr = 3e-4, weight_decay = 2e-4)
        
        scheduler = CosineAnnealingLR(optimizer, T_max = 200, eta_min = 3e-7)
            
        # move model to GPU
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        model = model.to(device)
        
        history = {
            "train_loss": [],
            # "val_loss" : [],
            "train_acc": [],
            # "val_acc" : []
        }
        
        # Training loops
        num_epochs = 200
        for epoch in range(num_epochs):
            print(f"Epoch {epoch}")
            if epoch == 0:
                print(f"First layer mean: {model.conv1.weight.data.mean():.6f}")
            for phase in ["train"]:
                if phase == "train":
                    model.train()
                else:
                    model.eval()
        
                running_loss = 0.0
                running_corrects = 0 # correct predictions
        
                for inputs, labels in tqdm(dataloaders[phase], leave = False):
                    inputs = inputs.to(device)
                    labels = labels.to(device)
        
                    optimizer.zero_grad() # clear the gradient from previous iteration
        
                    with torch.set_grad_enabled(phase == "train"):
                        outputs = model(inputs)
                        _, preds = torch.max(outputs, 1)
                        loss = criterion(outputs, labels) # check if output and labels match
        
                        if phase == "train":
                            loss.backward()
                            optimizer.step()
                            # scheduler.step() # scheduler here if OneCycleLR
        
                    running_loss += loss.item() * inputs.size(0)
                    running_corrects += (preds == labels).sum().item()
        
                epoch_loss = running_loss / dataset_sizes[phase]
                epoch_acc = running_corrects / dataset_sizes[phase]
                
                history[f"{phase}_loss"].append(epoch_loss)
                history[f"{phase}_acc"].append(epoch_acc)
        
                print(f"{phase} Loss: {epoch_loss:.7f} Acc: {epoch_acc:.7f}")
        
            scheduler.step()
            print(f"Epoch {epoch} LR: {scheduler.get_last_lr()[0]:.10f}")

            # if epoch % 15 == 0:
            #     torch.save({
            #         "epoch": epoch,
            #         "model_state_dict": model.state_dict(),
            #         "optimizer_state_dict": optimizer.state_dict(),
            #         "scheduler_state_dict": scheduler.state_dict(),
            #         "loss": epoch_loss,
            #         "history": history,
            #     }, os.path.join(workspace, f"weights/SSR/Experiment_2/seed{seed}_epoch{epoch}.pth"))
            
        print("Training complete!")
        
        train_loss.append(history["train_loss"])
        # val_loss.append(history["val_loss"])
        train_acc.append(history["train_acc"])
        # val_acc.append(history["val_acc"])
        
        torch.save(model.state_dict(), os.path.join(workspace, f"weights/SSR/OOD/wo_aug/rn18_seed{seed}_b16_rm_{excluded_label_name}.pth"))

    train_loss = np.array(train_loss)
    # val_loss = np.array(val_loss)
    train_acc = np.array(train_acc)
    # val_acc = np.array(val_acc)

1345
Excluding label 0 (2s1) from synthetic dataset
Training Run 0: seed 42
Epoch 0
First layer mean: -0.000035


train Loss: 0.7045040 Acc: 0.7574722
Epoch 0 LR: 0.0002999815
Epoch 1


train Loss: 0.0422313 Acc: 0.9923143
Epoch 1 LR: 0.0002999261
Epoch 2


train Loss: 0.0513769 Acc: 0.9888984
Epoch 2 LR: 0.0002998336
Epoch 3


train Loss: 0.0262722 Acc: 0.9940222
Epoch 3 LR: 0.0002997043
Epoch 4


...

train Loss: 0.0001094 Acc: 1.0000000
Epoch 195 LR: 0.0000005957
Epoch 196


train Loss: 0.0000356 Acc: 1.0000000
Epoch 196 LR: 0.0000004664
Epoch 197


train Loss: 0.0000442 Acc: 1.0000000
Epoch 197 LR: 0.0000003739
Epoch 198


train Loss: 0.0000321 Acc: 1.0000000
Epoch 198 LR: 0.0000003185
Epoch 199


train Loss: 0.0004739 Acc: 1.0000000
Epoch 199 LR: 0.0000003000
Training complete!
